# Phase 3: Kaggle Inference (vLLM Ensemble + Majority Vote)

**NBME Pipeline — NBME Score Clinical Patient Notes**

This notebook is designed to run on **Kaggle** (not Colab) with:
- **Accelerator**: GPU T4 x2 (or P100)
- **Internet**: OFF (competition requirement)
- **Datasets attached**: `nbme-score-clinical-patient-notes` + your `my-adapters` dataset

**Output**: `/kaggle/working/submission.csv`

---

**To use on Colab instead**, change `CONFIG` paths:
```python
"DATA_DIR":    Path("."),          # where your CSV files are
"ADAPTER_DIR": Path("./adapters"), # your adapters folder
"OUTPUT_DIR":  Path("."),
```

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# On Kaggle with internet OFF, these must be pre-installed or added as datasets.
# On Colab (internet ON), uncomment and run:
# !pip install -q "vllm>=0.9.0" "transformers>=5.5.0" "peft>=0.15.0" "rapidfuzz" "xgrammar"

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Configuration

> **Note**: Paths below are set for Kaggle. Adjust if running on Colab.

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import contextlib
import gc
import json
import logging
import re
import shutil
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from rapidfuzz.fuzz import partial_ratio_alignment
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoTokenizer,
)
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

# NOTE: destroy_model_parallel is still the correct call in vLLM v0.9 / v0.19.x
# to clear distributed state before loading a new model in the same process.
try:
    from vllm.distributed.parallel_state import destroy_model_parallel
except ImportError:
    # Fallback for future vLLM versions that may rename this
    def destroy_model_parallel():
        pass

# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # ── Kaggle paths ───────────────────────────────────────────────────────────
    "DATA_DIR":        Path("/kaggle/input/nbme-score-clinical-patient-notes"),
    "ADAPTER_DIR":     Path("/kaggle/input/my-adapters"),
    "OUTPUT_DIR":      Path("/kaggle/working"),

    # ── vLLM ──────────────────────────────────────────────────────────────────
    # enforce_eager=True is REQUIRED: disables CUDA graphs so GPU memory is
    # fully released when the engine is destroyed between models.
    # Without it, ~3.4 GiB of compiled Triton kernels remain resident (vLLM #36973).
    "ENFORCE_EAGER":      True,
    "GPU_MEM_UTIL":       0.85,
    "MAX_MODEL_LEN":      1024,   # notes ≤ 950 chars ≈ 300 tokens; 1024 is safe
    "MAX_NEW_TOKENS":     300,
    "LLM_TEMPERATURE":    0.0,    # greedy — maximum label consistency

    # ── Regex FSM constraint ───────────────────────────────────────────────────
    "MAX_SPANS_PER_FEATURE": 10,  # cap on number of spans in JSON output

    # ── Majority voting ────────────────────────────────────────────────────────
    "VOTE_THRESHOLD": 2,          # span accepted iff ≥ this many models agree

    # ── Span localization ──────────────────────────────────────────────────────
    "FUZZY_SCORE_CUTOFF": 70.0,   # minimum rapidfuzz score to accept a span match

    # ── Seed ──────────────────────────────────────────────────────────────────
    "SEED": 42,
}

# ── Model registry ─────────────────────────────────────────────────────────────
# Each entry describes one SLM: base model for merging + adapter location.
#
# dtype notes (must match training precision from Phase 2):
#   Qwen3.5-4B   → float16  (T4 native FP16 tensor cores; no overflow risk)
#   Gemma 4 E*B  → bfloat16 (mandatory — fp16 overflows in Gemma4 attention)
#
# model_class notes:
#   Gemma 4 is a VLM → AutoModelForImageTextToText  (text-only forward pass used)
MODEL_REGISTRY = [
    {
        "name":         "qwen_35_4b",
        "model_id":     "Qwen/Qwen3.5-4B",
        "model_class":  "causal_lm",
        "dtype":        torch.float16,
        "vllm_dtype":   "float16",
        "adapter_path": Path("/kaggle/input/my-adapters/qwen_35_4b_adapter"),
    },
    {
        "name":         "gemma_4_e2b",
        "model_id":     "google/gemma-4-E2B-it",
        "model_class":  "image_text_to_text",
        "dtype":        torch.bfloat16,
        "vllm_dtype":   "bfloat16",
        "adapter_path": Path("/kaggle/input/my-adapters/gemma_4_e2b_adapter"),
    },
    {
        "name":         "gemma_4_e4b",
        "model_id":     "google/gemma-4-E4B-it",
        "model_class":  "image_text_to_text",
        "dtype":        torch.bfloat16,
        "vllm_dtype":   "bfloat16",
        "adapter_path": Path("/kaggle/input/my-adapters/gemma_4_e4b_adapter"),
    },
]

# Shared system prompt — identical to Phases 1 and 2 for consistency
SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation.\n"
    "Output format: {\"spans\": [\"exact text 1\", \"exact text 2\"]}"
)

# =============================================================================
# LOGGING
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print('Configuration loaded ✓')

## SECTION 2 — LORA ADAPTER MERGER

In [ ]:
def merge_adapter_to_disk(
    model_spec: dict,
    output_dir: Path,
) -> Path:
    """
    Merge a LoRA adapter into the base model weights on CPU, then save the
    result to output_dir.

    Why merge (not LoRARequest):
      • Saves VRAM: vLLM's LoRA management buffers are not allocated.
      • Deterministic: avoids vLLM issue #5148 (output drift with dynamic LoRA).
      • device_map='cpu' for merging prevents OOM on T4 during this step.

    Returns the path to the merged model directory.
    """
    model_id     = model_spec["model_id"]
    model_class  = model_spec["model_class"]
    adapter_path = model_spec["adapter_path"]
    dtype        = model_spec["dtype"]

    merged_path = output_dir / f"merged_{model_spec['name']}"
    if merged_path.exists() and (merged_path / "config.json").exists():
        log.info(f"  [{model_spec['name']}] Merged model already on disk → {merged_path}")
        return merged_path

    log.info(f"  [{model_spec['name']}] Loading base model on CPU for merging …")

    # ── Load base model on CPU (no VRAM used during merge) ───────────────────
    load_kwargs = dict(
        pretrained_model_name_or_path = model_id,
        torch_dtype                   = dtype,
        device_map                    = "cpu",
    )
    if model_class == "causal_lm":
        base_model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
    else:
        base_model = AutoModelForImageTextToText.from_pretrained(**load_kwargs)

    # ── Load PEFT adapter and merge ───────────────────────────────────────────
    log.info(f"  [{model_spec['name']}] Merging LoRA adapter from {adapter_path} …")
    peft_model    = PeftModel.from_pretrained(base_model, str(adapter_path))
    merged_model  = peft_model.merge_and_unload()  # in-place merge, returns base model
    log.info(f"  [{model_spec['name']}] Merge complete. Saving to {merged_path} …")

    merged_path.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(merged_path))

    # Save tokenizer alongside model (needed for vLLM auto-detection)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.save_pretrained(str(merged_path))

    # ── Free CPU RAM before next operation ───────────────────────────────────
    del base_model, peft_model, merged_model, tokenizer
    gc.collect()
    log.info(f"  [{model_spec['name']}] Saved ✓")

    return merged_path

## SECTION 3 — PROMPT BUILDER

In [ ]:
def build_chat_prompt(
    feature_text: str,
    pn_history:   str,
    tokenizer:    AutoTokenizer,
) -> str:
    """
    Apply the model's chat template to produce the raw string prompt.
    Called once per (note, feature) pair.

    Includes /no_think to suppress Qwen3.5 chain-of-thought output,
    which must NOT appear in the JSON output consumed by the FSM.
    """
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": (
            f"Note: \"{pn_history.strip()}\"\n"
            f"Feature: {feature_text}\n\n"
            f"/no_think"
        )},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize            = False,
        add_generation_prompt = True,   # adds the assistant-turn opener
    )

## SECTION 4 — vLLM ENGINE LIFECYCLE

In [ ]:
def init_engine(merged_path: Path, model_spec: dict, cfg: dict) -> LLM:
    """
    Initialize a vLLM LLM instance from a merged (adapter-fused) model.

    enforce_eager=True is MANDATORY for sequential model loading:
      Without it, vLLM keeps compiled CUDA graphs (~3.4 GB) resident in the
      CUDA context after del+gc, causing OOM when the next model is loaded.
      (Known issue: vllm-project/vllm#36973)
    """
    log.info(f"  [{model_spec['name']}] Initialising vLLM engine …")
    llm = LLM(
        model                  = str(merged_path),
        dtype                  = model_spec["vllm_dtype"],
        gpu_memory_utilization = cfg["GPU_MEM_UTIL"],
        max_model_len          = cfg["MAX_MODEL_LEN"],
        enforce_eager          = cfg["ENFORCE_EAGER"],  # ← critical for cleanup
        trust_remote_code      = False,
        seed                   = cfg["SEED"],
    )
    log.info(f"  [{model_spec['name']}] vLLM engine ready.")
    return llm


def destroy_engine(llm: LLM, model_name: str) -> None:
    """
    Completely destroy a vLLM engine and free ALL GPU memory.

    Steps (order matters):
      1. destroy_model_parallel()  — clears distributed process groups
      2. destroy_process_group()   — clears torch.distributed state
      3. del llm                   — triggers Python __del__ on the engine
      4. gc.collect()              — force Python GC
      5. cuda.empty_cache()        — return freed blocks to CUDA allocator
      6. cuda.synchronize()        — ensure all CUDA ops have completed

    With enforce_eager=True, this reliably frees the full model allocation.
    """
    log.info(f"  [{model_name}] Destroying vLLM engine and freeing GPU memory …")

    # Step 1 – distributed state
    destroy_model_parallel()

    # Step 2 – torch distributed (suppress if single-GPU, no process group)
    with contextlib.suppress(Exception):
        torch.distributed.destroy_process_group()

    # Steps 3-6
    del llm
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
        total_gb = torch.cuda.mem_get_info()[1] / 1024**3
        log.info(
            f"  [{model_name}] VRAM after cleanup: "
            f"{free_gb:.1f} GB free / {total_gb:.1f} GB total"
        )

## SECTION 5 — INFERENCE RUNNER (one model at a time)

In [ ]:
def _parse_json_output(raw_text: str) -> list:
    """
    Parse the vLLM output string into a list of span strings.

    Handles Qwen3.5 think-token leakage even when /no_think is set,
    and gracefully falls back to empty list on malformed JSON.
    """
    # Strip any <think>...</think> block that may have leaked through
    raw_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL).strip()

    try:
        parsed = json.loads(raw_text)
        spans  = parsed.get("spans", [])
        return [s.strip() for s in spans if isinstance(s, str) and s.strip()]
    except (json.JSONDecodeError, AttributeError):
        # Attempt to recover a JSON fragment from the output
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group())
                spans  = parsed.get("spans", [])
                return [s.strip() for s in spans if isinstance(s, str) and s.strip()]
            except json.JSONDecodeError:
                pass
        return []


def run_inference_for_model(
    llm:          LLM,
    test_rows:    pd.DataFrame,
    pn_map:       dict,
    feat_map:     dict,
    tokenizer:    AutoTokenizer,
    cfg:          dict,
    model_name:   str,
) -> list:
    """
    Run inference for ALL test rows using a single loaded vLLM engine.

    For each row we:
      1. Build the chat prompt string
      2. Build a per-row regex from the patient note (FSM constraint)
      3. Batch all (prompt, SamplingParams) pairs through vLLM.generate()
      4. Parse the JSON output into a list of span strings

    Returns
    -------
    all_spans : list[list[str]]
        Outer list: one entry per test row (same order as test_rows).
        Inner list: extracted span texts for that row (may be empty).
    """
    log.info(f"  [{model_name}] Building prompts and regex patterns …")

    prompts      = []
    params_list  = []

    for _, row in test_rows.iterrows():
        pn_history   = pn_map.get(row["pn_num"], "").replace("\n", " ").strip()
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")

        # ── Prompt ───────────────────────────────────────────────────────────
        prompt = build_chat_prompt(feature_text, pn_history, tokenizer)
        prompts.append(prompt)

        # ── Per-row FSM regex ─────────────────────────────────────────────────
        # The regex is compiled from THIS note's character vocabulary,
        # physically preventing the LLM from hallucinating out-of-note text.
        regex   = build_constraint_regex(pn_history, cfg["MAX_SPANS_PER_FEATURE"])
        guided  = GuidedDecodingParams(regex=regex, backend="xgrammar")

        params = SamplingParams(
            temperature     = cfg["LLM_TEMPERATURE"],
            max_tokens      = cfg["MAX_NEW_TOKENS"],
            guided_decoding = guided,   # ← GuidedDecodingParams (vLLM v0.9+/v0.19+)
        )
        params_list.append(params)

    log.info(f"  [{model_name}] Running vLLM inference on {len(prompts)} rows …")

    # vLLM accepts a list of SamplingParams (one per prompt) for heterogeneous decoding
    outputs = llm.generate(
        prompts        = prompts,
        sampling_params = params_list,   # list[SamplingParams]
    )

    # ── Parse results ─────────────────────────────────────────────────────────
    all_spans = []
    for output in tqdm(outputs, desc=f"  [{model_name}] Parsing", leave=False):
        raw_text = output.outputs[0].text.strip() if output.outputs else ""
        spans    = _parse_json_output(raw_text)
        all_spans.append(spans)

    n_nonempty = sum(1 for s in all_spans if s)
    log.info(
        f"  [{model_name}] Inference complete. "
        f"Non-empty predictions: {n_nonempty}/{len(all_spans)}"
    )
    return all_spans

## SECTION 6 — CHARACTER-LEVEL MAJORITY VOTING

In [ ]:
def spans_to_char_array(
    span_locations: list,
    note_len:       int,
) -> np.ndarray:
    """
    Convert a list of (start, end) span tuples into a binary character array.

    Parameters
    ----------
    span_locations : list of (start, end) int tuples (end is exclusive)
    note_len       : length of the patient note in characters

    Returns
    -------
    np.ndarray of shape (note_len,) with dtype uint8; 1 inside spans, 0 outside.
    """
    arr = np.zeros(note_len, dtype=np.uint8)
    for start, end in span_locations:
        start = max(0, start)
        end   = min(note_len, end)
        if start < end:
            arr[start:end] = 1
    return arr


def char_array_to_spans(arr: np.ndarray) -> list:
    """
    Convert a binary character array back into (start, end) span tuples.

    Contiguous runs of 1s become a single span.
    Leading/trailing whitespace positions are trimmed from each span.

    Returns
    -------
    list of (start, end) tuples (end is exclusive, Python slice convention).
    """
    spans  = []
    n      = len(arr)
    i      = 0
    while i < n:
        if arr[i] == 1:
            start = i
            while i < n and arr[i] == 1:
                i += 1
            end = i
            spans.append((start, end))
        else:
            i += 1
    return spans


def locate_span_in_note(
    span_text:  str,
    pn_history: str,
    score_cutoff: float = 70.0,
) -> Optional[tuple]:
    """
    Map a span text string to its (start, end) character offset in pn_history.

    Strategy:
      1. Exact substring match (fastest, most accurate)
      2. Case-insensitive exact match (handles tokenization artefacts)
      3. rapidfuzz partial_ratio_alignment (handles minor word-boundary drifts)

    Uses partial_ratio_alignment which is O(N·M/64) in C++ — much faster than
    a Python sliding window while returning exact offset coordinates.

    Returns (start, end) or None if no match above score_cutoff.
    """
    span_text = span_text.strip()
    if not span_text or not pn_history:
        return None

    # ── Strategy 1: Exact ────────────────────────────────────────────────────
    idx = pn_history.find(span_text)
    if idx != -1:
        return (idx, idx + len(span_text))

    # ── Strategy 2: Case-insensitive exact ───────────────────────────────────
    idx = pn_history.lower().find(span_text.lower())
    if idx != -1:
        return (idx, idx + len(span_text))

    # ── Strategy 3: rapidfuzz partial_ratio_alignment ────────────────────────
    # Returns a ScoreAlignment namedtuple:
    #   .score, .src_start, .src_end, .dest_start, .dest_end
    # dest_* indices are into pn_history (the longer string).
    result = partial_ratio_alignment(
        span_text,
        pn_history,
        score_cutoff = score_cutoff,
    )
    if result is not None:
        return (result.dest_start, result.dest_end)

    return None


def character_level_majority_vote(
    model_predictions: list,      # list[list[list[str]]]  — [model][row][spans]
    test_rows:         pd.DataFrame,
    pn_map:            dict,
    vote_threshold:    int   = 2,
    fuzzy_cutoff:      float = 70.0,
) -> list:
    """
    Merge predictions from N models using character-level majority voting.

    Algorithm:
      For each test row:
        1. For each model, map its span texts → (start,end) offsets → binary char array
        2. Sum the N binary arrays element-wise → vote count per character
        3. Accept characters where vote count ≥ vote_threshold (default: 2 out of 3)
        4. Extract contiguous character runs → final (start, end) span list

    This is the same approach used by top NBME competition solutions:
      • Handles misaligned spans from different models gracefully
      • Avoids hard-span matching failures when models differ by a word or two
      • Naturally produces merged / extended spans when models partially overlap

    Parameters
    ----------
    model_predictions : list of per-model predictions
        model_predictions[m][i] = list of span text strings for row i, model m
    test_rows          : test.csv DataFrame
    pn_map             : pn_num → pn_history text
    vote_threshold     : minimum number of models that must cover a character
    fuzzy_cutoff       : rapidfuzz score threshold for span localization

    Returns
    -------
    final_spans : list[list[tuple[int,int]]]
        One (start, end) list per test row.
    """
    n_models = len(model_predictions)
    n_rows   = len(test_rows)

    log.info(
        f"Running character-level majority vote "
        f"({n_models} models, threshold={vote_threshold}/{n_models}) …"
    )

    final_spans = []

    for seq_idx, (_, row) in enumerate(tqdm(
        test_rows.iterrows(),
        total=n_rows,
        desc="Majority vote",
    )):
        pn_history = pn_map.get(row["pn_num"], "")
        note_len   = len(pn_history)

        if note_len == 0:
            final_spans.append([])
            continue

        # ── Accumulate character votes across all models ───────────────────────
        vote_array = np.zeros(note_len, dtype=np.int8)

        for model_idx in range(n_models):
            span_texts = model_predictions[model_idx][seq_idx]

            # Map each span text to a character offset, build binary array
            locations = []
            for text in span_texts:
                loc = locate_span_in_note(text, pn_history, fuzzy_cutoff)
                if loc is not None:
                    locations.append(loc)

            if locations:
                vote_array += spans_to_char_array(locations, note_len)

        # ── Apply threshold ───────────────────────────────────────────────────
        consensus_array = (vote_array >= vote_threshold).astype(np.uint8)

        # ── Whitespace suppression ────────────────────────────────────────────
        # Whitespace characters at span boundaries are never part of valid
        # NBME annotations — trim them from the consensus result.
        for i, ch in enumerate(pn_history):
            if ch in (' ', '\t', '\n', '\r') and consensus_array[i]:
                # Only suppress if at the very edge of a span run
                is_start = (i == 0 or consensus_array[i - 1] == 0)
                is_end   = (i == note_len - 1 or consensus_array[i + 1] == 0)
                if is_start or is_end:
                    consensus_array[i] = 0

        final_spans.append(char_array_to_spans(consensus_array))

    non_empty = sum(1 for s in final_spans if s)
    log.info(f"Majority vote complete. Non-empty rows: {non_empty}/{n_rows}")
    return final_spans

## SECTION 7 — SUBMISSION FORMATTER

In [ ]:
def format_location_string(spans: list, pn_history: str) -> str:
    """
    Format a list of (start, end) spans into the Kaggle submission string.

    Kaggle format: "start end;start end" (semicolon-separated, no brackets).

    Additional cleanup:
      • Strip leading/trailing whitespace from each span's character slice
        (the consensus char array may include boundary whitespace)
      • Deduplicate overlapping spans
      • Return NaN (empty string) if no valid spans

    Examples
    --------
    [(10, 25), (40, 55)]  →  "10 25;40 55"
    []                    →  "" (submission convention for no-annotation)
    """
    if not spans:
        return ""

    # ── Strip whitespace from span boundaries ─────────────────────────────────
    clean_spans = []
    for start, end in sorted(spans):
        # Walk inward past whitespace
        while start < end and pn_history[start] in (' ', '\t', '\n', '\r'):
            start += 1
        while end > start and pn_history[end - 1] in (' ', '\t', '\n', '\r'):
            end -= 1
        if start < end:
            clean_spans.append((start, end))

    # ── Merge overlapping / adjacent spans ────────────────────────────────────
    merged = []
    for start, end in sorted(clean_spans):
        if merged and start <= merged[-1][1]:
            # Overlapping or adjacent — extend the previous span
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    if not merged:
        return ""

    return ";".join(f"{s} {e}" for s, e in merged)


def build_submission(
    final_spans: list,
    test_df:     pd.DataFrame,
    pn_map:      dict,
) -> pd.DataFrame:
    """
    Build the submission DataFrame from per-row span lists.

    Returns a DataFrame with columns: id, location
    """
    rows = []
    for row_idx, (_, test_row) in enumerate(test_df.iterrows()):
        pn_history = pn_map.get(test_row["pn_num"], "")
        spans      = final_spans[row_idx] if row_idx < len(final_spans) else []
        location   = format_location_string(spans, pn_history)
        rows.append({
            "id":       test_row["id"],
            "location": location if location else np.nan,
        })

    return pd.DataFrame(rows)


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    log.info("=" * 65)
    log.info("  PHASE 3: Kaggle Inference")
    log.info("=" * 65)

    # ── Load data ─────────────────────────────────────────────────────────────
    log.info("Loading test data …")
    data_dir = cfg["DATA_DIR"]
    test_df  = pd.read_csv(data_dir / "test.csv")
    pn_df    = pd.read_csv(data_dir / "patient_notes.csv")
    feat_df  = pd.read_csv(data_dir / "features.csv")

    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = (
        feat_df
        .set_index(["case_num", "feature_num"])["feature_text"]
        .to_dict()
    )

    log.info(f"Test rows: {len(test_df)}")

    # ── Temp directory for merged models (auto-cleaned on exit) ───────────────
    tmp_root = cfg["OUTPUT_DIR"] / "merged_models"
    tmp_root.mkdir(parents=True, exist_ok=True)

    # ── Sequential inference loop ─────────────────────────────────────────────
    all_model_predictions = []   # [model_idx] → list[list[str]]

    for model_spec in MODEL_REGISTRY:
        model_name = model_spec["name"]
        log.info(f"\n{'='*65}")
        log.info(f"  Processing model: {model_name}")
        log.info(f"{'='*65}")

        # ── Step 1: Merge LoRA adapter into base model (CPU, no VRAM used) ───
        merged_path = merge_adapter_to_disk(model_spec, tmp_root)

        # ── Step 2: Load tokenizer (for chat template formatting) ─────────────
        tokenizer = AutoTokenizer.from_pretrained(str(merged_path))
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id

        # ── Step 3: Init vLLM engine ──────────────────────────────────────────
        llm = init_engine(merged_path, model_spec, cfg)

        # ── Step 4: Generate predictions for ALL test rows ────────────────────
        model_spans = run_inference_for_model(
            llm          = llm,
            test_rows    = test_df,
            pn_map       = pn_map,
            feat_map     = feat_map,
            tokenizer    = tokenizer,
            cfg          = cfg,
            model_name   = model_name,
        )
        all_model_predictions.append(model_spans)

        # ── Step 5: DESTROY vLLM engine — critical for sequential loading ─────
        destroy_engine(llm, model_name)
        del llm, tokenizer
        gc.collect()

        # ── Remove merged model from disk to free space for the next merge ────
        shutil.rmtree(str(merged_path), ignore_errors=True)
        log.info(f"  [{model_name}] Merged model deleted from disk.")

    # ── Character-level majority voting ───────────────────────────────────────
    log.info("\nStarting majority vote …")
    final_spans = character_level_majority_vote(
        model_predictions = all_model_predictions,
        test_rows         = test_df,
        pn_map            = pn_map,
        vote_threshold    = cfg["VOTE_THRESHOLD"],
        fuzzy_cutoff      = cfg["FUZZY_SCORE_CUTOFF"],
    )

    # ── Format and save submission ─────────────────────────────────────────────
    submission_df = build_submission(final_spans, test_df, pn_map)
    out_path = cfg["OUTPUT_DIR"] / "submission.csv"
    submission_df.to_csv(out_path, index=False)

    log.info("\n" + "=" * 65)
    log.info(f"  Submission saved → {out_path}")
    log.info(f"  Shape: {submission_df.shape}")
    log.info(f"  Non-empty predictions: "
             f"{submission_df['location'].notna().sum()} / {len(submission_df)}")
    log.info(f"\n{submission_df.head(10).to_string()}")
    log.info("=" * 65)
    log.info("Phase 3 complete ✓")


if __name__ == "__main__":
    main()

## Run Phase 3 — Generate Submission

In [ ]:
# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    log.info("=" * 65)
    log.info("  PHASE 3: Kaggle Inference")
    log.info("=" * 65)

    # ── Load data ─────────────────────────────────────────────────────────────
    log.info("Loading test data …")
    data_dir = cfg["DATA_DIR"]
    test_df  = pd.read_csv(data_dir / "test.csv")
    pn_df    = pd.read_csv(data_dir / "patient_notes.csv")
    feat_df  = pd.read_csv(data_dir / "features.csv")

    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = (
        feat_df
        .set_index(["case_num", "feature_num"])["feature_text"]
        .to_dict()
    )

    log.info(f"Test rows: {len(test_df)}")

    # ── Temp directory for merged models (auto-cleaned on exit) ───────────────
    tmp_root = cfg["OUTPUT_DIR"] / "merged_models"
    tmp_root.mkdir(parents=True, exist_ok=True)

    # ── Sequential inference loop ─────────────────────────────────────────────
    all_model_predictions = []   # [model_idx] → list[list[str]]

    for model_spec in MODEL_REGISTRY:
        model_name = model_spec["name"]
        log.info(f"\n{'='*65}")
        log.info(f"  Processing model: {model_name}")
        log.info(f"{'='*65}")

        # ── Step 1: Merge LoRA adapter into base model (CPU, no VRAM used) ───
        merged_path = merge_adapter_to_disk(model_spec, tmp_root)

        # ── Step 2: Load tokenizer (for chat template formatting) ─────────────
        tokenizer = AutoTokenizer.from_pretrained(str(merged_path))
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id

        # ── Step 3: Init vLLM engine ──────────────────────────────────────────
        llm = init_engine(merged_path, model_spec, cfg)

        # ── Step 4: Generate predictions for ALL test rows ────────────────────
        model_spans = run_inference_for_model(
            llm          = llm,
            test_rows    = test_df,
            pn_map       = pn_map,
            feat_map     = feat_map,
            tokenizer    = tokenizer,
            cfg          = cfg,
            model_name   = model_name,
        )
        all_model_predictions.append(model_spans)

        # ── Step 5: DESTROY vLLM engine — critical for sequential loading ─────
        destroy_engine(llm, model_name)
        del llm, tokenizer
        gc.collect()

        # ── Remove merged model from disk to free space for the next merge ────
        shutil.rmtree(str(merged_path), ignore_errors=True)
        log.info(f"  [{model_name}] Merged model deleted from disk.")

    # ── Character-level majority voting ───────────────────────────────────────
    log.info("\nStarting majority vote …")
    final_spans = character_level_majority_vote(
        model_predictions = all_model_predictions,
        test_rows         = test_df,
        pn_map            = pn_map,
        vote_threshold    = cfg["VOTE_THRESHOLD"],
        fuzzy_cutoff      = cfg["FUZZY_SCORE_CUTOFF"],
    )

    # ── Format and save submission ─────────────────────────────────────────────
    submission_df = build_submission(final_spans, test_df, pn_map)
    out_path = cfg["OUTPUT_DIR"] / "submission.csv"
    submission_df.to_csv(out_path, index=False)

    log.info("\n" + "=" * 65)
    log.info(f"  Submission saved → {out_path}")
    log.info(f"  Shape: {submission_df.shape}")
    log.info(f"  Non-empty predictions: "
             f"{submission_df['location'].notna().sum()} / {len(submission_df)}")
    log.info(f"\n{submission_df.head(10).to_string()}")
    log.info("=" * 65)
    log.info("Phase 3 complete ✓")


if __name__ == "__main__":
    main()

main()